# Notebook 01 — Univariate Linear Regression & Computational Graphs

**Session:** Linear Regression — L1, L2, and Huber Losses (Level 1)  
**Scheduled Date:** 2026-11-23 · **Module:** 02 — Calculus and Linear Algebra

---

## 1. Overview and Objectives

In this notebook, we implement univariate linear regression from elementary computational principles:
1. Model hypothesis: $\hat{y} = w \cdot x + b$.
2. Deconstruct forward propagation into discrete computational graph nodes.
3. Compute exact analytical derivatives using the backward chain rule.
4. Execute an exact numerical trace matching the reference single-sample SGD updates.
5. Simulate iterative stochastic gradient descent and plot error reduction curves.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Configure matplotlib for publication-quality figures
plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

## 2. The Computational Graph Formulation

Consider single-sample training with input feature $x$, target scalar $y$, weight $w$, and bias $b$.
We decompose the forward pass into four operations:
- **Node 1 (Multiplication):** $t = w \cdot x$
- **Node 2 (Addition / Affine Output):** $\hat{y} = t + b$
- **Node 3 (Residual):** $r = \hat{y} - y$
- **Node 4 (Squared Loss):** $L = r^2 = (\hat{y} - y)^2$

### Backward Chain Rule Derivation
Applying the chain rule from the loss $L$ back to the parameters $w$ and $b$:
$$\frac{\partial L}{\partial r} = 2r = 2(\hat{y} - y)$$
$$\frac{\partial r}{\partial \hat{y}} = 1, \quad \frac{\partial \hat{y}}{\partial t} = 1, \quad \frac{\partial \hat{y}}{\partial b} = 1$$
$$\frac{\partial t}{\partial w} = x$$

Multiplying local Jacobians yields:
$$\frac{\partial L}{\partial w} = \frac{\partial L}{\partial r} \cdot \frac{\partial r}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial t} \cdot \frac{\partial t}{\partial w} = 2(\hat{y} - y) \cdot 1 \cdot 1 \cdot x = 2x(\hat{y} - y)$$
$$\frac{\partial L}{\partial b} = \frac{\partial L}{\partial r} \cdot \frac{\partial r}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial b} = 2(\hat{y} - y) \cdot 1 \cdot 1 = 2(\hat{y} - y)$$


In [ ]:
class LinearRegressionNodeGraph:
    """Explicit node-by-node computational graph for univariate linear regression."""

    def __init__(self, w: float, b: float):
        self.w = float(w)
        self.b = float(b)

    def forward(self, x: float, y: float) -> tuple[float, float, float, float]:
        """Executes forward pass through elementary nodes."""
        self.x = float(x)
        self.y = float(y)
        self.t = self.w * self.x  # Node 1: multiplication
        self.y_hat = self.t + self.b  # Node 2: addition
        self.r = self.y_hat - self.y  # Node 3: residual
        self.loss = self.r**2  # Node 4: squared error
        return self.t, self.y_hat, self.r, self.loss

    def backward(self) -> tuple[float, float]:
        """Executes backward pass using chain rule."""
        dl_dr = 2.0 * self.r
        dr_dy_hat = 1.0
        dy_hat_dt = 1.0
        dy_hat_db = 1.0
        dt_dw = self.x

        # Chain rule combinations
        dl_dy_hat = dl_dr * dr_dy_hat
        dl_dw = dl_dy_hat * dy_hat_dt * dt_dw
        dl_db = dl_dy_hat * dy_hat_db
        return dl_dw, dl_db

    def step(self, lr: float) -> None:
        """Performs parameter update step."""
        dl_dw, dl_db = self.backward()
        self.w -= lr * dl_dw
        self.b -= lr * dl_db

## 3. Exact Numerical Trace

We verify the implementation against the reference trace:
- Initial weights: $w^{(0)} = -0.34$, $b^{(0)} = 0.04$
- Input point: $x = 6.7$, $y = 9.1$
- Learning rate: $\eta = 0.01$

Expected step 0:
- $t^{(0)} = -0.34 \times 6.7 = -2.278$
- $\hat{y}^{(0)} = -2.278 + 0.04 = -2.238$
- Residual $r^{(0)} = -2.238 - 9.1 = -11.338$
- Loss $L^{(0)} = (-11.338)^2 \approx 128.549$
- Gradients: $\frac{\partial L}{\partial w} = 2 \times 6.7 \times (-11.338) = -151.9292$, $\frac{\partial L}{\partial b} = 2 \times (-11.338) = -22.676$

Expected step 1:
- $w^{(1)} = -0.34 - 0.01 \times (-151.9292) = 1.179292$
- $b^{(1)} = 0.04 - 0.01 \times (-22.676) = 0.26676$
- $t^{(1)} = 1.179292 \times 6.7 \approx 7.901$
- $\hat{y}^{(1)} = 7.901 + 0.26676 \approx 8.168$
- Loss $L^{(1)} = (8.168 - 9.1)^2 = (-0.932)^2 \approx 0.868$


In [ ]:
# Instantiate graph with initial values
graph = LinearRegressionNodeGraph(w=-0.34, b=0.04)
x_val, y_val = 6.7, 9.1
lr = 0.01

# Step 0: Initial evaluation
t0, y_hat0, r0, loss0 = graph.forward(x_val, y_val)
dw0, db0 = graph.backward()

print(f"Step 0 Forward: t={t0:.4f}, y_hat={y_hat0:.4f}, residual={r0:.4f}, loss={loss0:.4f}")
print(f"Step 0 Backward: dL/dw={dw0:.4f}, dL/db={db0:.4f}")

assert np.isclose(t0, -2.278, atol=1e-3)
assert np.isclose(y_hat0, -2.238, atol=1e-3)
assert np.isclose(loss0, 128.55, atol=1e-1)
assert np.isclose(dw0, -151.9292, atol=1e-3)
assert np.isclose(db0, -22.6760, atol=1e-3)

# Step 1: Update parameters
graph.step(lr=lr)
print(f"\nAfter update: w={graph.w:.6f}, b={graph.b:.5f}")
assert np.isclose(graph.w, 1.179292, atol=1e-4)
assert np.isclose(graph.b, 0.26676, atol=1e-4)

# Step 1 Forward pass
t1, y_hat1, r1, loss1 = graph.forward(x_val, y_val)
print(f"Step 1 Forward: t={t1:.4f}, y_hat={y_hat1:.4f}, loss={loss1:.4f}")
assert np.isclose(t1, 7.901, atol=1e-2)
assert np.isclose(loss1, 0.868, atol=1e-2)
print("\nNumerical assertions passed successfully!")

## 4. Multi-Iteration Convergence on Single Sample

We now run 30 iterations of stochastic gradient descent on $(x=6.7, y=9.1)$ to observe the
exponential decay of error and asymptotic convergence toward zero loss.


In [ ]:
history = {"iteration": [], "w": [], "b": [], "loss": [], "y_hat": []}
model = LinearRegressionNodeGraph(w=-0.34, b=0.04)

for it in range(30):
    _, y_hat, _, loss = model.forward(x_val, y_val)
    history["iteration"].append(it)
    history["w"].append(model.w)
    history["b"].append(model.b)
    history["loss"].append(loss)
    history["y_hat"].append(y_hat)
    model.step(lr=0.01)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history["iteration"], history["loss"], marker="o", color="tab:red", lw=1.5)
ax1.set_title("Loss vs. Iteration (Single-Sample SGD)")
ax1.set_xlabel("Iteration")
ax1.set_ylabel("Squared Loss L")
ax1.set_yscale("log")

ax2.plot(history["iteration"], history["w"], label="Weight w", color="tab:blue", lw=1.5)
ax2.plot(history["iteration"], history["b"], label="Bias b", color="tab:green", lw=1.5)
ax2.set_title("Parameter Trajectories")
ax2.set_xlabel("Iteration")
ax2.set_ylabel("Value")
ax2.legend()

plt.tight_layout()
plt.show()

## 5. Geometric Visualization of Candidate Regression Lines

Visualizing how the candidate line $\hat{y} = w x + b$ pivots and shifts across iterations to
intersect the target training point $(x=6.7, y=9.1)$.


In [ ]:
x_space = np.linspace(2, 10, 100)
iterations_to_plot = [0, 1, 2, 5, 29]

plt.figure(figsize=(8, 5))
plt.scatter(
    [x_val], [y_val], color="red", s=100, zorder=5, label=f"Target Sample ({x_val}, {y_val})"
)

for it in iterations_to_plot:
    w_curr = history["w"][it]
    b_curr = history["b"][it]
    y_line = w_curr * x_space + b_curr
    plt.plot(x_space, y_line, label=f"Iter {it}: w={w_curr:.2f}, b={b_curr:.2f}", alpha=0.85)

plt.title("Hypothesis Line Adaptation under Stochastic Updates")
plt.xlabel("Feature x (Area)")
plt.ylabel("Target y (Price)")
plt.legend()
plt.tight_layout()
plt.show()

## 6. Summary

- Univariate linear regression maps inputs through an affine transformation: $\hat{y} = wx + b$.
- The computational graph breaks computation into atomic forward and backward operations.
- Single-sample SGD updates parameters along the negative gradient direction: $\mathbf{\theta} \leftarrow \mathbf{\theta} - \eta \nabla L$.
- A single update with appropriate learning rate ($\eta=0.01$) reduces the squared error on the target sample by over 99%.
